In [1]:
import os
import random
import numpy as np
import pandas as pd

import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader

import cv2
from torchvision import models


# Reproducibility (score-neutral/stability)
def seed_everything(seed=42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = False
    torch.backends.cudnn.benchmark = True


seed_everything(42)

DATA_ROOT = "/kaggle/input/cassava-leaf-disease-classification"
TRAIN_CSV = os.path.join(DATA_ROOT, "train.csv")
SAMPLE_SUB = os.path.join(DATA_ROOT, "sample_submission.csv")
TRAIN_IMG_DIR = os.path.join(DATA_ROOT, "train_images")
TEST_IMG_DIR = os.path.join(DATA_ROOT, "test_images")

assert os.path.exists(TRAIN_CSV), f"Missing: {TRAIN_CSV}"
assert os.path.exists(SAMPLE_SUB), f"Missing: {SAMPLE_SUB}"
assert os.path.isdir(TRAIN_IMG_DIR), f"Missing dir: {TRAIN_IMG_DIR}"
assert os.path.isdir(TEST_IMG_DIR), f"Missing dir: {TEST_IMG_DIR}"

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
device



device(type='cuda')

In [2]:
# Minimal torchvision-like preprocessing without relying on missing build_transforms
IMG_SIZE = 512  # reasonable for time/accuracy tradeoff; can be increased if time allows

IMAGENET_MEAN = np.array([0.485, 0.456, 0.406], dtype=np.float32)
IMAGENET_STD = np.array([0.229, 0.224, 0.225], dtype=np.float32)


def preprocess_bgr_image(img_bgr, train=False):
    # img_bgr: uint8 HxWx3
    img = cv2.cvtColor(img_bgr, cv2.COLOR_BGR2RGB)

    if train:
        # light augmentations to improve generalization, keeping core approach simple
        if random.random() < 0.5:
            img = cv2.flip(img, 1)
        if random.random() < 0.3:
            img = cv2.flip(img, 0)

    img = cv2.resize(img, (IMG_SIZE, IMG_SIZE), interpolation=cv2.INTER_AREA)

    img = img.astype(np.float32) / 255.0
    img = (img - IMAGENET_MEAN) / IMAGENET_STD
    img = np.transpose(img, (2, 0, 1))  # CHW
    return torch.from_numpy(img)


class CassavaTrainDataset(Dataset):
    def __init__(self, df, img_dir, train=True):
        self.df = df.reset_index(drop=True)
        self.img_dir = img_dir
        self.train = train

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        fn = row["image_id"]
        y = int(row["label"])
        path = os.path.join(self.img_dir, fn)
        img = cv2.imread(path)
        if img is None:
            raise FileNotFoundError(f"Could not read image: {path}")
        x = preprocess_bgr_image(img, train=self.train)
        return x, torch.tensor(y, dtype=torch.long)


class CassavaTestDataset(Dataset):
    def __init__(self, image_ids, img_dir):
        self.image_ids = list(image_ids)
        self.img_dir = img_dir

    def __len__(self):
        return len(self.image_ids)

    def __getitem__(self, idx):
        fn = self.image_ids[idx]
        path = os.path.join(self.img_dir, fn)
        img = cv2.imread(path)
        if img is None:
            raise FileNotFoundError(f"Could not read image: {path}")
        x = preprocess_bgr_image(img, train=False)
        return x, fn




In [3]:
# Load data
train_df = pd.read_csv(TRAIN_CSV)
sample_df = pd.read_csv(SAMPLE_SUB)

# Basic checks
assert set(train_df.columns) == {"image_id", "label"}
assert set(sample_df.columns) == {"image_id", "label"}

num_classes = train_df["label"].nunique()
assert num_classes == 5, f"Expected 5 classes, got {num_classes}"

# Train/val split (stratified)
from sklearn.model_selection import StratifiedShuffleSplit

splitter = StratifiedShuffleSplit(n_splits=1, test_size=0.1, random_state=42)
train_idx, val_idx = next(splitter.split(train_df["image_id"], train_df["label"]))
tr_df = train_df.iloc[train_idx].reset_index(drop=True)
va_df = train_df.iloc[val_idx].reset_index(drop=True)

len(tr_df), len(va_df), tr_df["label"].value_counts().sort_index().to_dict()



(16848, 1873, {0: 845, 1: 1711, 2: 1882, 3: 10370, 4: 2040})

In [4]:
# Dataloaders
BATCH_SIZE = 32
NUM_WORKERS = 2  # conservative for Kaggle notebooks

train_ds = CassavaTrainDataset(tr_df, TRAIN_IMG_DIR, train=True)
val_ds = CassavaTrainDataset(va_df, TRAIN_IMG_DIR, train=False)

train_loader = DataLoader(
    train_ds,
    batch_size=BATCH_SIZE,
    shuffle=True,
    num_workers=NUM_WORKERS,
    pin_memory=torch.cuda.is_available(),
    drop_last=True,
)
val_loader = DataLoader(
    val_ds,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=NUM_WORKERS,
    pin_memory=torch.cuda.is_available(),
    drop_last=False,
)

# Model: ResNet50 classifier head (standard baseline; replaces missing custom model)
model = models.resnet50(weights=models.ResNet50_Weights.IMAGENET1K_V2)
model.fc = nn.Linear(model.fc.in_features, 5)
model = model.to(device)

# Loss/optim
criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.AdamW(model.parameters(), lr=3e-4, weight_decay=1e-4)

# Simple scheduler (stable)
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=2)



Downloading: "https://download.pytorch.org/models/resnet50-11ad3fa6.pth" to /root/.cache/torch/hub/checkpoints/resnet50-11ad3fa6.pth
100%|██████████| 97.8M/97.8M [00:00<00:00, 113MB/s]


In [5]:
# Train a small number of epochs to fit within time (no early stopping, fixed epochs)
EPOCHS = 2


def accuracy_from_logits(logits, y):
    preds = logits.argmax(dim=1)
    return (preds == y).float().mean().item()


for epoch in range(1, EPOCHS + 1):
    model.train()
    tr_loss = 0.0
    tr_acc = 0.0
    n_tr = 0

    for xb, yb in train_loader:
        xb = xb.to(device, non_blocking=True)
        yb = yb.to(device, non_blocking=True)

        optimizer.zero_grad(set_to_none=True)
        logits = model(xb)
        loss = criterion(logits, yb)
        loss.backward()
        optimizer.step()

        bs = xb.size(0)
        tr_loss += loss.item() * bs
        tr_acc += accuracy_from_logits(logits.detach(), yb) * bs
        n_tr += bs

    scheduler.step()

    model.eval()
    va_loss = 0.0
    va_acc = 0.0
    n_va = 0
    with torch.no_grad():
        for xb, yb in val_loader:
            xb = xb.to(device, non_blocking=True)
            yb = yb.to(device, non_blocking=True)
            logits = model(xb)
            loss = criterion(logits, yb)

            bs = xb.size(0)
            va_loss += loss.item() * bs
            va_acc += accuracy_from_logits(logits, yb) * bs
            n_va += bs

    print(
        f"Epoch {epoch}/{EPOCHS} | "
        f"train loss {tr_loss/n_tr:.4f} acc {tr_acc/n_tr:.4f} | "
        f"val loss {va_loss/n_va:.4f} acc {va_acc/n_va:.4f}"
    )



Epoch 1/2 | train loss 0.5121 acc 0.8258 | val loss 0.4054 acc 0.8644
Epoch 2/2 | train loss 0.3607 acc 0.8774 | val loss 0.3349 acc 0.8911


In [6]:
# Inference aligned to sample_submission order to guarantee correct length/order
test_image_ids = sample_df["image_id"].tolist()
test_ds = CassavaTestDataset(test_image_ids, TEST_IMG_DIR)
test_loader = DataLoader(
    test_ds,
    batch_size=64,
    shuffle=False,
    num_workers=NUM_WORKERS,
    pin_memory=torch.cuda.is_available(),
    drop_last=False,
)

model.eval()
pred_map = {}
with torch.no_grad():
    for xb, fns in test_loader:
        xb = xb.to(device, non_blocking=True)
        logits = model(xb)
        preds = logits.argmax(dim=1).detach().cpu().numpy().astype(int)
        for fn, p in zip(fns, preds):
            pred_map[fn] = int(p)

# Build submission from sample_submission to enforce exact row count/order
sub = sample_df.copy()
sub["label"] = sub["image_id"].map(pred_map).astype(int)

# Safety checks to avoid "Invalid submission length"
assert len(sub) == len(sample_df)
assert sub["label"].notna().all(), "Some test images missing predictions"
sub.head()



,image_id,label
0,1234294272.jpg,3
1,1234332763.jpg,3
2,1234375577.jpg,1
3,1234555380.jpg,1
4,1234571117.jpg,3


In [7]:
# Write submission
out_path = "./submission.csv"
sub.to_csv(out_path, index=False)

# Verify format quickly
check = pd.read_csv(out_path)
assert list(check.columns) == ["image_id", "label"]
assert len(check) == len(sample_df)
print(out_path, "written with shape:", check.shape)
print(check.head(3).to_string(index=False))

./submission.csv written with shape: (2676, 2)
      image_id  label
1234294272.jpg      3
1234332763.jpg      3
1234375577.jpg      1
